In [4]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

import torch
from torch import nn
import torch.nn.functional as F
import math

from transformers import AutoTokenizer, AutoModelForCausalLM

# from linear_transformer import patch_model_for_lvp

In [1]:
import torch

class OuterProdSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:
        orig_dtype = x.dtype
        x_f = x.to(dtype) # Safer to use .to() instead of .float() if dtype arg is passed
        
        # Standard stable Softmax
        x_max = x_f.max(dim=dim, keepdim=True).values
        exps = torch.exp(x_f - x_max)
        sum_exps = torch.sum(exps, dim=dim, keepdim=True)
        s = exps / sum_exps
        
        # Save tensors and attributes needed for backward
        ctx.save_for_backward(x, s.to(orig_dtype))
        ctx._orig_dtype = orig_dtype
        ctx._dtype = dtype
        ctx._dim = dim  # Save the dimension!
        
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # The incoming vector t from the next layer
    ) -> tuple[torch.Tensor, None, None, None]: 
        # Note: Return a None for every input argument in forward (x, dim, dtype)
        
        (x, s) = ctx.saved_tensors
        dim = ctx._dim
        dtype = ctx._dtype
        
        x_f = x.to(dtype)
        s_f = s.to(dtype)
        grad_t_f = grad_t.to(dtype)

        s_dot_t = torch.sum(s_f * grad_t_f, dim=dim, keepdim=True)
        
        norm_sq = torch.sum(x_f * x_f, dim=dim, keepdim=True)
        norm_sq = torch.clamp(norm_sq, min=1e-12)
        
        result = (s_dot_t / norm_sq) * x_f

        return result.to(ctx._orig_dtype), None, None

In [43]:
torch.matmul(((_a.unsqueeze(-1) * a.unsqueeze(-2)) / (torch.norm(a, dim=-1, keepdim=True)**2).unsqueeze(-1)).T, b).shape

torch.Size([64, 64, 64])

In [2]:
a = torch.rand(12, 64)
# a[:, 20:30] = 0
a.requires_grad = True
b = torch.rand(12, 64)

_a = OuterProdSoftmax.apply(a, -1)
loss = (_a * b).sum()
print(loss)

loss.backward()
loss_at_x = (a * a.grad).sum()
print(loss_at_x)



tensor(6.1880, grad_fn=<SumBackward0>)
tensor(6.1880, grad_fn=<SumBackward0>)


In [11]:
import torch

# 1. Setup the initial vectors
x = torch.tensor([2.0, 1.0, -1.0, 0.5])  # Pre-softmax logits (x_i)
s = torch.softmax(x, dim=0)              # Softmax output (s_i)
t = torch.tensor([0.0, 1.0, 0.0, 0.0])   # Example target unembedding vector (t)

# 2. Compute the squared L2 norm: ||x||^2
# Using dot product is mathematically identical to sum of squares
norm_sq = torch.dot(x, x)  
# print(norm_sq, (x * x[None, :]).sum(), torch.norm(x)**2)

# 3. Construct the local linear surrogate matrix M: (s * x^T) / ||x||^2
M =  s[:, None] * x[None, :] / norm_sq
# print(torch.outer(s, x), s.T * x[:, None])
# print(M, x)

# --- VERIFYING THE FORWARD PASS ---
# Equation: M @ x = s
reconstructed_s = M @ x

print("Original Softmax (s):   ", s)
print("Reconstructed (M @ x):  ", reconstructed_s)
print("Matches?                ", torch.allclose(s, reconstructed_s))

# --- EXECUTING THE BACKWARD PASS ---
# Passing t back through the linearized operation using the adjoint (transpose)
# Equation: x_back = M^T @ t
x_back = M.T @ t

print("\nBackward pass vector:   ", x_back)

# --- VERIFYING THE CONSERVATION PROPERTY ---
# Equation: s • t = x • x_back
forward_dot = torch.dot(s, t)
backward_dot = torch.dot(x, x_back)

print(f"\nForward Dot (s • t):      {forward_dot.item():.6f}")
print(f"Backward Dot (x • x_back): {backward_dot.item():.6f}")
print("Property holds?           ", torch.allclose(forward_dot, backward_dot))

Original Softmax (s):    tensor([0.6095, 0.2242, 0.0303, 0.1360])
Reconstructed (M @ x):   tensor([0.6095, 0.2242, 0.0303, 0.1360])
Matches?                 True

Backward pass vector:    tensor([ 0.0717,  0.0359, -0.0359,  0.0179])

Forward Dot (s • t):      0.224208
Backward Dot (x • x_back): 0.224208
Property holds?            True


In [ ]:
a = torch.tensor([1, 2, 3], dtype=torch.float)
b = torch.tensor([4, 5, 6], dtype=torch.float)
print(a, b, '\n')

a_exp = torch.exp(a)
s = a_exp / a_exp.sum()
print(s, '\n')

norm_sq = torch.dot(a, a)  
print(norm_sq, (a * a[None, :]).sum(), torch.norm(a)**2)

_s = s[:, None] * a[None, :] / (torch.norm(a) ** 2)

_s = _s @ a
print(_s, '\n')

s0 = a_exp[1] / a_exp.sum() # * (1 - a_exp[0]/ a_exp.sum()) * (1 - a_exp[2] / a_exp.sum())
print(s0)

l = (s * b).sum()
print(l)


tensor([1., 2., 3.]) tensor([4., 5., 6.]) 

tensor([0.0900, 0.2447, 0.6652]) 

tensor(14.) tensor(14.) tensor(14.0000)
tensor([0.0900, 0.2447, 0.6652]) 

tensor(0.2447)
tensor(5.5752)


In [ ]:
torch.norm(s), torch.norm(a)

(tensor(0.7145), tensor(1.4142))

In [18]:
3 ** 2 * torch.var(a), 3 * torch.norm(s)

(tensor(9.), tensor(2.1436))

In [78]:
from math import exp

a = a_sum = exp(2)
ap = a / a_sum

b_sum = (a_sum + exp(1))
bp = exp(1) / b_sum
ap = ap * (1 - bp)


c_sum = b_sum + exp(3)
cp = exp(3) / c_sum
ap = ap * (1 - cp)
bp = bp * (1 - cp)

print(ap, bp, cp, ap + bp + cp)





0.24472847105479764 0.09003057317038045 0.6652409557748219 1.0


In [ ]:

class LinearAdd(torch.autograd.Function):

    @staticmethod
    def forward(
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
        other: torch.Tensor,  # (..., d) or scalar
        eps: float
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        x_f = x.float()
        other_f = other.float()
        
        add_out = x_f + other_f
        
        safe_x = torch.where(x_f >= 0, x_f.clamp(min=eps), x_f.clamp(max=-eps))
        add_ratio = add_out / safe_x

        ctx.save_for_backward(add_ratio.to(orig_dtype))
        ctx._orig_dtype = orig_dtype

        return add_out.to(orig_dtype)

    @staticmethod
    def backward(
        ctx: torch.autograd.function.FunctionCtx,
        grad: torch.Tensor,  # (..., d)
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        (add_ratio,) = ctx.saved_tensors
        add_ratio_f = add_ratio.float()
        grad_f = grad.float()
        
        grad_x = grad_f * add_ratio_f

        return grad_x.to(ctx._orig_dtype), None, 


class LinearSub(torch.autograd.Function):

    @staticmethod
    def forward(
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
        other: torch.Tensor,  # (..., d) or scalar
        eps: float
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        x_f = x.float()
        other_f = other.float()
        
        add_out = x_f - other_f
        
        safe_x = torch.where(x_f >= 0, x_f.clamp(min=eps), x_f.clamp(max=-eps))
        sub_ratio = add_out / safe_x

        ctx.save_for_backward(sub_ratio.to(orig_dtype))
        ctx._orig_dtype = orig_dtype

        return add_out.to(orig_dtype)

    @staticmethod
    def backward(
        ctx: torch.autograd.function.FunctionCtx,
        grad: torch.Tensor,  # (..., d)
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        (sub_ratio,) = ctx.saved_tensors
        sub_ratio_f = sub_ratio.float()
        grad_f = grad.float()
        
        grad_x = grad_f * sub_ratio_f

        return grad_x.to(ctx._orig_dtype), None, None

In [147]:
a = torch.rand(12, 64)
# a[:, 20:30] = 0
a.requires_grad = True
b = torch.rand(12, 64)

s = torch.tensor(5.0)

_a = LinearSub.apply(a, s, 1e-7)
loss = (_a * b).sum()
print(loss)

loss.backward()
loss_at_x = (a * a.grad).sum()
print(loss_at_x)



tensor(-1756.3942, grad_fn=<SumBackward0>)
tensor(-1756.3942, grad_fn=<SumBackward0>)


In [52]:
class DTDSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        ctx.save_for_backward(s.to(orig_dtype), x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        (s,x) = ctx.saved_tensors
        dim = ctx._dim
        dtype = ctx._dtype
        s_f = s.to(dtype=dtype)
        # f⁻¹(t) = t - s · (Σ_j t_j)
        result = grad_t.to(dtype=dtype) - s_f * (grad_t.to(dtype=dtype).sum(dim=dim, keepdim=True))
        return result.to(ctx._orig_dtype), None, None  # no gradient w.r.t. dim
    
class SecantSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        ctx.save_for_backward(s.to(orig_dtype), x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        (s, x) = ctx.saved_tensors
        dim = ctx._dim
        dtype = ctx._dtype
        s_f = s.to(dtype=dtype)
        x_f = x.to(dtype=dtype)
        result = s_f / (x_f + 1e-07) * grad_t.to(dtype=dtype)
        return result.to(ctx._orig_dtype), None, None  # no gradient w.r.t. dim

class PosRationSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        ctx.save_for_backward(x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        (x, ) = ctx.saved_tensors
        dtype = ctx._dtype
        x_f = x.to(dtype=dtype).clip(min=0.0)
        result = x_f / x_f.sum(dim=-1, keepdim=True) * grad_t.to(dtype=dtype)
        return result.to(ctx._orig_dtype), None, None  # no gradient w.r.t. dim
    
class RatioSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        ctx.save_for_backward(s.to(orig_dtype), x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        (s, x) = ctx.saved_tensors
        dtype = ctx._dtype
        x_f = x.to(dtype=dtype)
        s_f = s.to(dtype=dtype)
        result = s_f * (s_f * grad_t.to(dtype=dtype)).sum(-1, keepdim=True) / (x_f + 1e-7)
        return result.to(ctx._orig_dtype), None, None  # no gradient w.r.t. dim
    
class IntegratedSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        
        ctx.save_for_backward(x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        (x,) = ctx.saved_tensors
        dtype = ctx._dtype
        dim = ctx._dim

        k_steps = 10
        
        grad_t_f = grad_t.to(dtype=dtype)
        accumulated_grads = torch.zeros_like(x, dtype=dtype)
        
        # Generate alphas: [1/k, 2/k, ..., 1.0]
        alphas = torch.linspace(1.0 / k_steps, 1.0, steps=k_steps, device=x.device, dtype=dtype)
        baseline = torch.zeros_like(x, dtype=x.dtype, device=x.device)
        for alpha in alphas:
            x_alpha = baseline + alpha * (x - baseline)
            s_alpha = torch.softmax(x_alpha, dim=dim, dtype=dtype)
            
            dot_product = (grad_t_f * s_alpha).sum(dim=dim, keepdim=True)
            step_grad = s_alpha * (grad_t_f - dot_product)
            accumulated_grads += step_grad
            
        # Average the accumulated gradients
        result = accumulated_grads / k_steps

        return result.to(ctx._orig_dtype), None, None
    

class SecantJacobianSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
        dtype: torch.dtype = torch.float32
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        s = torch.softmax(x, dim=dim, dtype=dtype)
        ctx.save_for_backward(s.to(orig_dtype), x)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        ctx._dtype = dtype
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None, None]:
        s, x = ctx.saved_tensors
        dim = ctx._dim
        dtype = ctx._dtype
        
        s_f = s.to(dtype=dtype)
        x_f = x.to(dtype=dtype)  # Fixed: previously mapped to 's'
        grad_f = grad_t.to(dtype=dtype)
        
        # 1. Define Baseline
        # DeepLIFT typically uses 0.0 as the neutral reference baseline
        x_prime = torch.zeros_like(x_f)
        
        # 2. Prevent Overflow (Log-Sum-Exp Trick)
        # Shift both actual and baseline inputs by the max value to prevent NaN
        M = torch.clamp(x_f.max(dim=dim, keepdim=True).values, min=0.0)
        x_shifted = x_f - M
        x_prime_shifted = x_prime - M
        
        exp_x = torch.exp(x_shifted)
        exp_x_prime = torch.exp(x_prime_shifted)
        
        # 3. Compute baseline summation metrics
        Z_prime = exp_x_prime.sum(dim=dim, keepdim=True)
        s_prime = exp_x_prime / Z_prime
        
        # 4. Vectorized Finite Difference 
        # Z_i calculates the exact denominator as if ONLY x_i changed from baseline
        Z_i = Z_prime - exp_x_prime + exp_x
        
        S_grad_t = (grad_f * exp_x_prime).sum(dim=dim, keepdim=True)
        B = (grad_f * s_prime).sum(dim=dim, keepdim=True)
        
        # The exact Vector-Jacobian Product (VJP) if only input i changes
        term1 = (S_grad_t + grad_f * (exp_x - exp_x_prime)) / Z_i
        delta_vjp = term1 - B
        delta_x = x_f - x_prime
        
        # 5. Continuous Fallback
        # Standard Softmax Jacobian VJP for when delta_x is almost zero
        cont_vjp = s_f * (grad_f - (grad_f * s_f).sum(dim=dim, keepdim=True))
        
        # 6. Apply discrete difference (Secant) safely
        epsilon = 1e-5
        secant_vjp = delta_vjp / (delta_x + 1e-9) # 1e-9 protects against division by absolute zero
        
        # Where x hasn't moved from baseline, use continuous derivative
        result = torch.where(torch.abs(delta_x) < epsilon, cont_vjp, secant_vjp)
        
        return result.to(ctx._orig_dtype), None, None

In [74]:
S = 10
x = torch.rand(5, 32, S, S) * 10
x[:, :, 2] = 0
# x = torch.tensor([1e-7, 0])
x.requires_grad = True
y = torch.rand(5, 32, S, S)
# y = torch.tensor([1.0, 1.0])

# _x = DTDSoftmax.apply(x)
_x = SecantJacobianSoftmax.apply(x)
# _x = IntegratedSoftmax.apply(x)
# _x = PosRationSoftmax.apply(x)
# _x = SecantSoftmax.apply(x)
# _x = F.softmax(x, dim=-1)

loss = (_x * y).sum()
loss.backward()

loss_at_x = (x * x.grad).sum()

print(loss.item(), loss_at_x.item())

794.9464721679688 4.9400739669799805


In [14]:
x.grad

tensor([2500000.2500, 5000000.0000])

In [10]:
x.grad

tensor([[[[9.5214e-02, 8.6052e-02, 9.1400e-02,  ..., 9.8072e-02,
           1.1154e-01, 1.6358e-01],
          [1.1016e-01, 7.9505e-02, 3.6781e+00,  ..., 8.4371e-02,
           9.9605e-01, 6.8463e-02],
          [4.6301e+05, 4.6301e+05, 4.6301e+05,  ..., 4.6301e+05,
           4.6301e+05, 4.6301e+05],
          ...,
          [3.2086e-01, 1.1047e-01, 2.1142e-01,  ..., 1.4708e-01,
           3.8492e-01, 1.5413e-01],
          [1.6877e-01, 1.0388e-01, 7.5796e-02,  ..., 9.3689e-02,
           6.1322e-02, 6.1377e-02],
          [9.2928e-02, 1.1453e-01, 5.9174e-01,  ..., 9.9744e-02,
           2.5527e-01, 1.2080e-01]],

         [[9.1685e-02, 9.2236e-02, 9.2780e-02,  ..., 1.5184e-01,
           2.1386e-01, 1.0411e-01],
          [7.2582e-02, 7.9773e-02, 6.6332e-02,  ..., 7.0287e-02,
           1.0102e-01, 6.1460e-02],
          [4.0708e+05, 4.0708e+05, 4.0708e+05,  ..., 4.0708e+05,
           4.0708e+05, 4.0708e+05],
          ...,
          [2.3036e-01, 8.9032e-02, 2.1644e-01,  ..., 8.7355

In [ ]:
_x.detach().grad_fn == None

In [5]:
from nnsight import NNsight

LLAMA_ID = "gpt2"
LLAMA_ID = "meta-llama/Llama-3.1-8B"
LLAMA_ID = "google/gemma-2-2b"
LLAMA_ID = "Qwen/Qwen2.5-0.5B"

tokenizer   = AutoTokenizer.from_pretrained(LLAMA_ID)
if not hasattr(tokenizer, 'pad_token') or not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation='eager'
).eval()
# lvp_config = {
#             "frozen_norm": True, "attn_act_fn": "frozen_denom_softmax",
#             "matmul_fn": "bilinear_matmul", "mul_fn": "bilinear_mul",
#             "mlp_act_fn": "secant_gelu_tanh", "attn_softcap_fn": "secant_tanh",
#         }
# model = patch_model_for_lvp(model, **lvp_config)
model = NNsight(model)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [10]:
model.model.layers[0].self_attn.q_proj.bias

Parameter containing:
tensor([-1.4221e-02,  2.6001e-02, -8.9844e-02, -1.2988e-01, -1.4500e+01,
         2.4902e-01,  3.0078e-01,  1.3770e-01, -1.5438e+01, -3.4000e+01,
        -1.5125e+01,  6.8438e+00, -4.3750e-01,  7.9062e+00,  5.9375e-01,
         2.0938e+00, -1.5688e+01,  4.6250e+00, -2.5000e-01, -7.8906e-01,
         3.2656e+00,  5.5469e-01,  5.7422e-01,  4.6875e+00, -2.8438e+00,
         2.5000e+00,  2.0000e+00, -1.3438e+00,  5.5176e-02,  1.1641e+00,
         8.0469e-01,  1.5156e+00,  4.9805e-02,  1.0938e-01,  1.6406e-01,
        -2.8906e-01, -1.2188e+01, -1.6113e-01, -3.5547e-01,  3.3984e-01,
         1.4125e+01,  6.1562e+00,  2.2500e+00, -1.6211e-01,  7.5073e-03,
        -2.8000e+01, -2.2754e-01, -3.8594e+00,  9.4375e+00, -1.8672e+00,
         1.6641e+00, -7.4219e-02,  3.7812e+00, -2.2344e+00, -3.4688e+00,
        -3.7656e+00,  4.6562e+00,  1.7031e+00,  1.5391e+00,  2.0625e+00,
         1.9219e+00,  1.9531e-01, -1.3984e+00,  3.5938e-01, -1.4688e+00,
        -2.1250e+00, -2.6172e

In [28]:
inputs = tokenizer(['Paris is the capital of', 'Berlin is the capital of'], return_tensors='pt', padding=True)

with model.trace(**inputs):
    
    for layer in model.model.layers:
        attn_scores = layer.self_attn.source.attention_interface_0.source.nn_functional_softmax_0.input
        attn_weights = layer.self_attn.source.attention_interface_0.source.to_0.output

        print(torch.var(attn_scores, dim=-1))
        raise



non cached init <nnsight 3801884985981793483>
tensor([[[    inf,     inf,     inf,     inf,     inf, 38.0000],
         [    inf,     inf,     inf,     inf,     inf,  3.0000],
         [    inf,     inf,     inf,     inf,     inf,  5.7812],
         [    inf,     inf,     inf,     inf,     inf,  5.0625],
         [    inf,     inf,     inf,     inf,     inf,  0.7422],
         [    inf,     inf,     inf,     inf,     inf,  3.6875],
         [    inf,     inf,     inf,     inf,     inf,  3.4062],
         [    inf,     inf,     inf,     inf,     inf,  1.9609]],

        [[    inf,     inf,     inf,     inf,     inf, 42.0000],
         [    inf,     inf,     inf,     inf,     inf,  3.6250],
         [    inf,     inf,     inf,     inf,     inf,  6.0938],
         [    inf,     inf,     inf,     inf,     inf,  5.1250],
         [    inf,     inf,     inf,     inf,     inf,  0.6641],
         [    inf,     inf,     inf,     inf,     inf,  3.4062],
         [    inf,     inf,     inf,     i

Traceback (most recent call last):
  File "/raid/conda/envs/dacslab_lasse/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3033278/220597043.py", line 3, in <module>
    with model.trace(**inputs):
  File "/tmp/ipykernel_3033278/220597043.py", line 10, in <module>
    raise
  File "<nnsight 3801884985981793483>", line 9, in __nnsight_tracer_3801884985981793483__

RuntimeError: No active exception to reraise


In [15]:
inputs = tokenizer(['something', 'something else'], return_tensors='pt', padding=True)

cache = {}
with model.trace(**inputs):
    for i, layer in enumerate(model.model.layers):
        cache[f"{i}.attn"] = layer.self_attn.output[1]

    loss = model.output.logits[:, -1].max(dim=-1).values.sum()
    with loss.backward():
        for i in range(model.config.num_hidden_layers - 1, -1, -1):
            print(cache[f"{i}.attn"][0, 1],'\n', cache[f"{i}.attn"].grad[0, 1], '\n\n')



non cached init <nnsight 8386458041731963446>
non cached init <nnsight 3651916920424497834>
tensor([[0.3340, 0.3340, 0.3340],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.8633, 0.1357]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>) 
 tensor([[0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0229, 0.0096, 0.0208]], device='cuda:0', dtype=torch.bfloat16) 


tensor([[0.3340, 0.3340, 0.3340],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.9219, 0.0796]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>) 
 tensor([[-0.4473, -0.4199,  4.6875],
        [-0.2197,  0.0481,  0.7812],
        [-2.8125,  1.1016,  1.6719]], device='cuda:0', dtype=torch.bfloat16) 


tensor([[0.3340, 0.3340, 0.3340],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.3711, 0.6289]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>) 
 tensor([[20.7500, -3.1250,  6.5000],
        [ 9.7500, -1.2578,  4.7500],
 

In [4]:

cache = {}

def rel_err(
    x: torch.Tensor,
    gx: torch.Tensor,
    y: torch.Tensor,
    gy: torch.Tensor,
) -> float:
    """Relative conservation error: |s_in - s_out| / |loss|."""
    s_in  = (x.float() * gx.float()).sum().item()
    s_out = (y.float() * gy.float()).sum().item()
    return abs(s_in - s_out) / abs(s_in + 1e-10)

with model.trace(**inputs):
    # _, attention_mask, position_embedding = 
    emb = model.model.embed_tokens.output.save()
    layer_input = model.model.layers[0].inputs[0].save()
    kwargs = model.model.layers[0].inputs[1].save()

    for i, layer in enumerate(model.model.layers):
        cache[f"{i}.in"] = layer.input_layernorm.input
        cache[f"{i}.ln1"] = layer.input_layernorm.output
        cache[f"{i}.attn"] = layer.self_attn.output[0]
        cache[f"{i}.mid"] = layer.post_attention_layernorm.input
        cache[f"{i}.ln2"] = layer.post_attention_layernorm.output
        cache[f"{i}.mlp"] = layer.mlp.output
        cache[f"{i}.out"] = layer.output
    cache['logits'] = model.lm_head.output.save()
    loss = model.output.logits[:, -1].max(dim=-1).values.sum()
    print(loss)

    with loss.backward():
        for i in range(len(model.model.layers) - 1, -1, -1):
            cache[f"{i}.out_grad"] = cache[f"{i}.out"].grad
            cache[f"{i}.mlp_grad"] = cache[f"{i}.mlp"].grad
            cache[f"{i}.ln2_grad"] = cache[f"{i}.ln2"].grad
            cache[f"{i}.mid_grad"] = cache[f"{i}.mid"].grad
            cache[f"{i}.attn_grad"] = cache[f"{i}.attn"].grad
            cache[f"{i}.ln1_grad"] = cache[f"{i}.ln1"].grad
            cache[f"{i}.in_grad"] = cache[f"{i}.in"].grad

            # cache[f"{i}.mid_grad"]
            # loss_at_i_out = (cache[f"{i}.out"] * cache[f"{i}.out"].grad).sum(-1)
            # out_grad = cache[f"{i}.out"].grad.save()
            # loss_at_i_post_mlp = (cache[f"{i}.mlp"] * cache[f"{i}.mlp"].grad).sum(-1)
            # loss_at_i_pre_mlp = (cache[f"{i}.ln2"] * cache[f"{i}.ln2"].grad).sum(-1)
            # loss_at_i_pre_ln2 = (cache[f"{i}.mid"] * (cache[f"{i}.mid"].grad - out_grad)).sum(-1)
            # loss_at_i_mid = (cache[f"{i}.mid"] * cache[f"{i}.mid"].grad).sum(-1)
            # print(f"Loss at layer {i} out: {loss_at_i_out.sum()}")
            # print(f"Loss at layer {i} post_mlp: {loss_at_i_post_mlp.sum()}")
            # print(f"Loss at layer {i} ln2_out: {loss_at_i_pre_mlp.sum()}")
            # print(f"Loss at layer {i} ln2_in: {loss_at_i_pre_ln2.sum()}")
            # print(f"Loss at layer {i} mid: {loss_at_i_mid.sum()}")
    
            print(f"mlp error: {rel_err(cache[f"{i}.ln2"], cache[f"{i}.ln2_grad"], cache[f"{i}.mlp"], cache[f"{i}.mlp_grad"]):.4f}")
            print(f"ln2 error: {rel_err(cache[f"{i}.mid"], (cache[f"{i}.mid_grad"] - cache[f"{i}.out_grad"]), cache[f"{i}.ln2"], cache[f"{i}.ln2_grad"]):.4f}")
            print(f"attn error: {rel_err(cache[f"{i}.ln1"], cache[f"{i}.ln1_grad"], cache[f"{i}.attn"], cache[f"{i}.attn_grad"]):.4f}")
            print(f"ln1 error: {rel_err(cache[f"{i}.in"], (cache[f"{i}.in_grad"] - cache[f"{i}.mid_grad"]), cache[f"{i}.ln1"], cache[f"{i}.ln1_grad"]):.4f}")


non cached init <nnsight 6836210784927040936>
tensor(56.5000, device='cuda:0', dtype=torch.bfloat16, grad_fn=<SumBackward0>)
non cached init <nnsight 3111826965681732056>
mlp error: 1.4105
ln2 error: 0.0324
attn error: 1.0012
ln1 error: 0.3307
mlp error: 16.0130
ln2 error: 0.4635
attn error: 1.0092
ln1 error: 1.0927
mlp error: 0.2642
ln2 error: 0.4340
attn error: 1.0203
ln1 error: 1.0418
mlp error: 0.6963
ln2 error: 0.0371
attn error: 0.9180
ln1 error: 1.0666
mlp error: 1.4947
ln2 error: 0.3033
attn error: 0.7068
ln1 error: 1.0262
mlp error: 3.1681
ln2 error: 0.0641
attn error: 0.6781
ln1 error: 1.0045
mlp error: 13.7782
ln2 error: 0.1585
attn error: 1.0208
ln1 error: 1.0123
mlp error: 10.9013
ln2 error: 10.9371
attn error: 0.9506
ln1 error: 1.0169
mlp error: 3.5426
ln2 error: 13.1865
attn error: 0.9270
ln1 error: 1.0161
mlp error: 0.6334
ln2 error: 0.1678
attn error: 0.8843
ln1 error: 1.0241
mlp error: 0.3271
ln2 error: 0.2022
attn error: 0.6807
ln1 error: 1.0155
mlp error: 3.9512
ln2

In [ ]:
x = torch.rand(1, 4, model.config.hidden_size, requires_grad=True).to(model.device)
y = torch.rand(1, 1, model.config.hidden_size).to(model.device)


i = 0
with model.model.layers[i].trace(layer_input[0].detach(), **kwargs):
    x_at_ln1 = model.model.layers[i].input_layernorm.output.save()
    print(model.model.layers[0].self_attn.source.eager_attention_forward_0.input)
    
    # x_at_attn = model.model.layers[i].self_attn.output[0].save()
    # x_at_ln2 = model.model.layers[i].post_attention_layernorm.output
    # x_at_mlp = model.model.layers[i].mlp.output
    # _x = model.model.layers[i].output
    # loss = (_x[:, -1] * y).sum(-1)
    # print(loss)

    # with loss.backward():
    #     loss_at_x = (x * x.grad).sum(-1)
    #     print(loss_at_x.sum())



In [ ]:
x_at_ln1.shape

In [4]:
from nnsight import NNsight

LLAMA_ID = "meta-llama/Llama-3.1-8B"
LLAMA_ID = "Qwen/Qwen2.5-0.5B"
LLAMA_ID = "google/gemma-2-2b"
LLAMA_ID = "gpt2"

tokenizer   = AutoTokenizer.from_pretrained(LLAMA_ID)
org_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_ID, dtype=torch.float, device_map="auto",
    attn_implementation='eager'
).eval()
org_model = NNsight(org_model)

inputs = tokenizer("Paris is the capital of", return_tensors='pt')
target_id = tokenizer.encode(' France', add_special_tokens=False)[0]

org_logits = org_model(inputs['input_ids'].to(org_model.device)).logits


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [5]:
with torch.no_grad(), org_model.trace(**inputs):
    layer = org_model.transformer.h[0]
    org_inp = layer.input.save()
    org_ln1 = layer.ln_1.output.save()
    # print(layer.attn.source)
    # print(layer.attn.source.attention_interface_0.source.attn_output_transpose_0.output.shape)
    org_c = layer.attn.source.attn_output_reshape_0.output.save()
    org_attn = layer.attn.output[0].save()
    org_attn_w = layer.attn.output[1].save()
    org_ln2 = layer.ln_2.output.save()
    # print(layer.mlp.source)
    org_mlp_p = layer.mlp.source.self_c_fc_0.output.save()
    org_mlp_m = layer.mlp.source.self_c_proj_0.output.save()
    org_mlp = layer.mlp.output.save()


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_ID, dtype=torch.float, device_map="auto", attn_implementation='eager'
).eval()
model = patch_model_for_lvp(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [7]:
patched_logits = model(inputs['input_ids'].to(model.device)).logits
with torch.no_grad(), model.trace(**inputs):
    layer = model.transformer.h[0]
    p_inp = layer.input.save()
    p_ln1 = layer.ln_1.output.save()
    p_c = layer.attn.source.attn_output_reshape_0.output.save()
    p_attn = layer.attn.output[0].save()
    p_attn_w = layer.attn.output[1].save()
    p_ln2 = layer.ln_2.output.save()
    p_mlp_p = layer.mlp.source.self_c_fc_0.output.save()
    p_mlp_m = layer.mlp.source.self_c_proj_0.output.save()
    p_mlp = layer.mlp.output.save()

In [11]:
model.transformer.h[0].ln_2.weight.data

tensor([0.1310, 0.2093, 0.2066, 1.2542, 1.2638, 1.2695, 0.0935, 0.0793, 0.2260,
        1.3008, 0.2324, 1.1525, 1.2761, 1.2695, 0.7216, 0.2881, 0.1325, 1.1616,
        1.2393, 0.1919, 1.2380, 1.2227, 1.1692, 0.1753, 1.1690, 1.3008, 0.1421,
        1.2539, 1.2142, 1.2007, 1.2299, 1.0236, 1.3710, 1.3308, 1.0197, 0.1152,
        0.2317, 1.2060, 1.2695, 1.0617, 1.2837, 1.2734, 1.2930, 1.3086, 0.3260,
        1.2463, 1.2305, 1.2773, 0.1372, 0.2164, 1.3143, 1.2393, 0.1665, 1.2103,
        0.3018, 0.3291, 1.0898, 0.1200, 1.2149, 0.1043, 0.1793, 1.2202, 0.1299,
        0.1555, 0.0643, 1.1776, 1.2422, 0.1817, 0.1472, 1.3810, 0.3341, 0.1125,
        1.2120, 1.1521, 1.3403, 0.9472, 0.1831, 0.1996, 0.2273, 1.0586, 0.3701,
        1.2456, 1.2461, 1.0780, 0.8484, 0.1608, 0.2123, 0.0573, 1.2763, 0.1909,
        1.2071, 1.2695, 1.2037, 1.2617, 1.2927, 1.3385, 1.3401, 0.1224, 1.3009,
        0.0869, 1.3700, 0.1047, 0.1528, 0.5143, 1.2446, 1.3158, 1.0508, 0.4896,
        1.2535, 1.2462, 0.1860, 1.3008, 

In [ ]:
p = F.softmax(org_logits[-1, 0, :], dim=-1)
q = F.softmax(patched_logits[-1, 0, :], dim=-1)
kl = (p * (p.log() - q.log())).sum().item()
kl

In [ ]:
org_model.transformer.h[0].ln_1(p_inp), model.transformer.h[0].ln_1(p_inp)

In [ ]:
from transformers.models.gpt2.modeling_gpt2 import ACT2FN
ACT2FN['gelu_new']

In [ ]:
org_model.transformer

In [ ]:
p_mlp_p, org_mlp_p

In [ ]:
p.topk(10).indices, q.topk(10).indices

In [ ]:
org_c.shape, p_c.shape

In [ ]:
torch.allclose(p_inp, org_inp), torch.allclose(p_ln1, org_ln1), torch.allclose(p_attn_w, org_attn_w), torch.allclose(p_attn, org_attn, atol=1e-6), torch.allclose(p_ln2, org_ln2, atol=1e-6), torch.allclose(p_mlp, org_mlp, atol=1e-4)

In [ ]:
p_ln1, org_ln1

In [ ]:
p_mlp, org_mlp

In [ ]:
torch.allclose(model.transformer.h[0].attn.c_proj.weight.data, org_model.transformer.h[0].attn.c_proj.weight.data)

In [ ]:
org_model.config

In [ ]:


with model.trace(**inputs):
    emb = model.transformer.wte.output
    last_layer_input = model.transformer.h[-1].input
    last_layer_post_attn = model.transformer.h[-1].mlp.input
    model_out = model.lm_head.input
    loss = model.lm_head.output[0, -1, target_id]
    print(f"Loss: {loss.item():.4f}")

    # Backward pass
    with loss.backward():

        loss_at_x = (model_out * model_out.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (last_layer_post_attn * last_layer_post_attn.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (last_layer_input * last_layer_input.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (emb * emb.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

In [ ]:
inputs = tokenizer("Paris is the capital of", return_tensors='pt')
target_id = tokenizer.encode('France', add_special_tokens=False)[0]

with model.trace(**inputs):
    emb = model.model.embed_tokens.output
    last_layer_input = model.model.layers[-1].input
    last_layer_post_attn = model.model.layers[-1].post_attention_layernorm.input
    model_out = model.lm_head.input
    loss = model.lm_head.output[0, -1, target_id]
    print(f"Loss: {loss.item():.4f}")

    # Backward pass
    with loss.backward():

        loss_at_x = (model_out * model_out.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (last_layer_post_attn * last_layer_post_attn.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (last_layer_input * last_layer_input.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

        loss_at_x = (emb * emb.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")

In [ ]:
model

In [ ]:
class CustomGELU(torch.autograd.Function):
    @staticmethod
    def forward(
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        x_f = x.float()
        ctx.save_for_backward(x_f)
        ctx._orig_dtype = orig_dtype
        return F.gelu(x).to(orig_dtype)

    @staticmethod
    def backward(
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        (x_f,) = ctx.saved_tensors
        c = 0.5 * (1.0 + torch.erf(x_f / math.sqrt(2.0)))  # Gaussian CDF
        return (grad_t.float() * c).to(ctx._orig_dtype)
    

class CustomSiLU(torch.autograd.Function):

    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        x_f = x.float()
        ctx.save_for_backward(x_f)
        ctx._orig_dtype = orig_dtype
        return F.silu(x_f).to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        (x_f,) = ctx.saved_tensors
        c = torch.sigmoid(x_f)
        return (grad_t.float() * c).to(ctx._orig_dtype)
    
class CustomSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., N)
        dim: int = -1,
    ) -> torch.Tensor:  # (..., N)
        orig_dtype = x.dtype
        x_f = x.float()
        s = torch.softmax(x_f, dim=dim)
        ctx.save_for_backward(s)
        ctx._orig_dtype = orig_dtype
        ctx._dim = dim
        return s.to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., N)
    ) -> tuple[torch.Tensor, None]:
        (s,) = ctx.saved_tensors
        dim = ctx._dim
        # f⁻¹(t) = t - s · (Σ_j t_j)
        result = grad_t.float() - s * grad_t.float().sum(dim=dim, keepdim=True)
        return result.to(ctx._orig_dtype), None  # no gradient w.r.t. dim
    

class CustomBilinear(torch.autograd.Function):

    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
        y: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        ctx.save_for_backward(x.float(), y.float())
        ctx._orig_dtype = orig_dtype
        return (x.float() * y.float()).to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., d)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        x_f, y_f = ctx.saved_tensors
        t_f = grad_t.float()
        grad_x = (0.5 * t_f * y_f).to(ctx._orig_dtype)
        grad_y = (0.5 * t_f * x_f).to(ctx._orig_dtype)
        return grad_x, grad_y
    
class CustomMatmulBilinear(torch.autograd.Function):

    @staticmethod
    def forward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        x: torch.Tensor,  # (..., d)
        y: torch.Tensor,  # (..., d)
    ) -> torch.Tensor:  # (..., d)
        orig_dtype = x.dtype
        ctx.save_for_backward(x.float(), y.float())
        ctx._orig_dtype = orig_dtype
        return torch.matmul(x.float(), y.float()).to(orig_dtype)

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: torch.autograd.function.FunctionCtx,
        grad_t: torch.Tensor,  # (..., d)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        x_f, y_f = ctx.saved_tensors
        t_f = grad_t.float()
        grad_x = (0.5 * torch.matmul(t_f, y_f.mT)).to(ctx._orig_dtype)
        grad_y = (0.5 * torch.matmul(x_f.mT, t_f)).to(ctx._orig_dtype)
        return grad_x, grad_y

In [ ]:
class CustomRMSNorm(nn.Module):
    def __init__(self, shape: tuple, eps: float) -> None:
        super().__init__()
        self.weight = torch.randn(shape)  # (d_model,) — reference to original parameter
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, N, d_model)
        x_f = x.float()

        rms = x_f.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()  # (B, N, 1)
        w = (self.weight.float() / rms)  # (B, N, d_model) — effective linear weight
        w = w.detach()

        out = (x_f * w).to(x.dtype)
        return out
    
class CustomMLP(nn.Module):

    def __init__(self, in_shape, hidden_shape) -> None:
        super().__init__()
        self.c_fc = nn.Linear(in_shape, hidden_shape, bias=False)    # W_up:   (d_model, d_ffn)
        self.c_proj = nn.Linear(hidden_shape, in_shape, bias=False)  # W_down: (d_ffn, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, N, d_model)
        h = self.c_fc(x)
        h_act = CustomGELU.apply(h)                         
        out = self.c_proj(h_act)
        return out
    
class CustomSwiGLU(nn.Module):
    def __init__(self, in_shape, hidden_shape) -> None:
        super().__init__()
        self.gate_proj = nn.Linear(in_shape, hidden_shape, bias=False)   # (d_ffn, d_model)
        self.up_proj = nn.Linear(in_shape, hidden_shape, bias=False)       # (d_ffn, d_model)
        self.down_proj = nn.Linear(hidden_shape, in_shape, bias=False)   # (d_model, d_ffn)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, N, d_model)
        gate_pre = self.gate_proj(x)  # (B, N, d_ffn) pre-activation
        gate = CustomSiLU.apply(gate_pre)                    # (B, N, d_ffn) G* = silu(gate_pre)
        up = self.up_proj(x)          # (B, N, d_ffn) U*
        hidden = CustomBilinear.apply(gate, up)
        out = self.down_proj(hidden)

        return out
    

class CustomAttention(nn.Module):

    def __init__(
        self,
        in_shape,
        num_heads,
        head_dim
    ) -> None:
        super().__init__()
        self.q_proj = nn.Linear(in_shape, num_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(in_shape, num_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(in_shape, num_heads * head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * head_dim, in_shape, bias=False)

        self.num_heads = num_heads      # H_q
        self.head_dim = head_dim        # d_head
        self.scale = 1.0 / math.sqrt(head_dim)

    def forward(
        self,
        x: torch.Tensor,                       # (B, N, d_model)
    ) -> torch.Tensor:                         # (B, N, d_model)
        N, _ = x.shape

        Q = self.q_proj(x).reshape(N, self.num_heads, self.head_dim).transpose(0, 1)
        K = self.k_proj(x).reshape(N, self.num_heads, self.head_dim).transpose(0, 1)
        V = self.v_proj(x).reshape(N, self.num_heads, self.head_dim).transpose(0, 1)

        A_scores = CustomMatmulBilinear.apply(Q, K.transpose(-2, -1)) * self.scale  # (B, H, N, N)
        A = CustomSoftmax.apply(A_scores, -1)  # (H, N, N) torch.softmax(A_scores, -1) #

        AV = CustomMatmulBilinear.apply(A, V)
        out = self.o_proj(AV.transpose(0,1).reshape(N, -1))

        return (out, A)

In [ ]:
class CustomModel(nn.Module):
    """Composite model: MLP -> SwiGLU -> Attention -> RMSNorm."""
    
    def __init__(self, d_model: int, d_ffn: int, num_heads: int, head_dim: int) -> None:
        super().__init__()
        self.mlp = CustomMLP(d_model, d_ffn)
        self.glu = CustomSwiGLU(d_model, d_ffn)
        self.attn = CustomAttention(d_model, num_heads, head_dim)
        self.norm = CustomRMSNorm(d_model, eps=1e-7)
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through all layers.
        
        Args:
            x: (B, N, d_model)
            
        Returns:
            out: (B, N, d_model) — normalized output
            A: (H, N, N) — attention weights
        """
        out = x + self.mlp(x)
        out = out + self.glu(out)
        attn_out, A = self.attn(out)
        out = out + attn_out
        out = self.norm(out)
        return out, A

In [ ]:
from nnsight import NNsight

# Initialize model and inputs
model = CustomModel(d_model=256, d_ffn=1024, num_heads=32, head_dim=8)
model = NNsight(model)
x = torch.randn(5, 256, requires_grad=True)
y_target = torch.randn(1, 256)

# Forward pass
with model.trace(x):

    mlp_out = model.mlp.output
    out, A = model.output

    # Compute loss: dot product of last token with target
    loss = torch.matmul(out[-1], y_target.T)
    print(f"Loss: {loss.item():.4f}")

    # Backward pass
    with loss.backward():
        mlp_grad = mlp_out.grad
        x_grad = x.grad

    with torch.no_grad():
        loss_at_mlp = (mlp_out * mlp_grad).sum()
        print(f"Loss at mlp: {loss_at_mlp.item():.4f}")

        loss_at_x = (x * x.grad).sum()
        print(f"Loss at x (x · ∇x): {loss_at_x.item():.4f}")